# Revision Workshop 2 — Revising `sklearn` functions


## Part 1 - `StandardScaler()` and `MinMaxScaler()`

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

stscaler = StandardScaler()
mmscaler = MinMaxScaler()

X1 = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15]])

X2 = stscaler.fit_transform(X1)

X3 = mmscaler.fit_transform(X1)

## Questions

1. What is `StandardScaler()` doing to `X1`? 

2. What is `MinMaxScaler()` doing to `X1`? 

3. Why would anyone bother to run `StandardScaler()` or `MinMaxScaler()`?

**Answer to the question above:**


1. **StandardScaler()** subtracts the mean and divides by the standard deviation for each feature, so values have mean 0 and std 1.

2. **MinMaxScaler()** rescales each feature to a chosen range (default 0–1), using  
   \(x' = (x - x_{min}) / (x_{max} - x_{min})\).

3. We scale data so all features are on comparable ranges — this helps gradient-based and distance-based models train faster and more reliably.


## Part 2 — Passing the wrong argument

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import Perceptron

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

model = Perceptron()
model.fit(X_train, y_train)

y_pred = model.predict(X_train)
accuracy_score(y_test, y_pred)

**Exercises**

1. Identify the bug.

2. Correct it.

3. Explain what was wrong.

**Answer to the question above:**

1. **Bug:** you’re predicting on the **training** data (`model.predict(X_train)`) but then comparing it to the **test** labels (`accuracy_score(y_test, y_pred)`).

2. **Correct version:**

   ```python
   y_pred = model.predict(X_test)
   accuracy_score(y_test, y_pred)
   ```

3. **What was wrong:** the features and labels must come from the **same split**. If you evaluate with test labels, you must also predict on the test features. Mixing train features with test labels gives a meaningless accuracy.


# Part 3 — Cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

cross_val_score(model.fit(X_train, y_train), X, y, cv=5)


**Exercises**

1. What is `cross_val_score` meant to do?

2. Why does this code fail?

3. Correct the code.

**Answers**

1. **`cross_val_score`** performs *k-fold cross-validation*: it splits the dataset into `cv` folds, trains the model on each training fold, tests it on the held-out fold, and returns all the scores.

2. The code fails because `model.fit(X_train, y_train)` returns a **fitted model**, not an unfitted estimator.  
   `cross_val_score` expects an *unfitted* estimator so it can fit it itself on each fold.

3. **Correct version:**
   ```python
   from sklearn.model_selection import cross_val_score

   scores = cross_val_score(model, X, y, cv=5)
   print(scores)
   print(scores.mean())


# Part 4 — Dimensional mismatch

Let's create an error:

In [ ]:
X.shape, y.shape

X_train.shape, y_train.shape

((442, 10), (442,))

In [ ]:
X_train = X[:5]
y_train = y[:10]

model.fit(X_train, y_train)

**Exercises**

1. Run the code above. Read the error message carefully. What does it tell you? What does it mean?
2. Can you fix the code?

**Answers**

1. It tells me `ValueError: Found input variables with inconsistent numbers of samples: [5, 10]`. It means you're using 5 datapoints in X and 10 datapoints in y, where it should be the same number

2. You can make both 5, both 10, or both whichever number.

# Part 5 - Reading documentation

Let's learn about the function `GridSearchCV`.

Read the documentation here:
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [31]:
from sklearn.model_selection import GridSearchCV

GridSearchCV

sklearn.model_selection._search.GridSearchCV

**Exercises**

1. What does `GridSearchCV` do?

2. Which arguments are required?

3. Which are optional?

4. What does it return?

**Answers**

1. **`GridSearchCV`** searches over combinations of hyperparameters to find the best-performing model using cross-validation.

2. **Required arguments:**
   - `estimator`: the model or pipeline to tune  
   - `param_grid`: a dictionary mapping parameter names to lists of values to try  

3. **Optional arguments (common ones):**
   - `cv`: number of cross-validation folds (default = 5)  
   - `scoring`: metric used to evaluate performance  
   - `n_jobs`: number of parallel jobs  
   - `verbose`, `refit`, `return_train_score`, etc.

4. **Returns:** a fitted `GridSearchCV` object containing:
   - `best_params_`: the parameter set with the best score  
   - `best_score_`: mean cross-validation score for that set  
   - `best_estimator_`: the model refitted on all training data with those parameters.
